# Gold 06A — Test Replay Validation

This notebook validates the trained model artifacts by replaying the saved model/configuration outputs against the held-out Gold test rows.

The goal is intentionally narrow:

1. Load the saved Gold 02 / 03A / 03B / 03C model artifacts and configuration artifacts.
2. Load the Gold 01 split data.
3. Re-run the saved baseline and cascade logic against the held-out test rows.
4. Compare the replayed test metrics against the metrics saved by the original model-training notebooks.
5. Save replayed row-level outputs and summary comparison tables for Gold 06B.

This notebook does **not** retrain models. It validates that the saved production-ready artifacts can be reused to produce similar held-out test behavior.


## 1. Imports and notebook context

This section keeps the setup close to the other Gold notebooks: standard Python imports, project context loading, logger/ledger setup, and artifact path resolution.


In [12]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Mapping, Sequence

import json
import logging
import math
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

from utils.core.file_io import load_json, save_json
from utils.core.notebook_context import load_notebook_context
from utils.core.logging_setup import configure_logging


In [13]:
# Shared notebook context.

CONTEXT_STAGE = "gold_model_replay_validation"
CONTEXT_DATASET = "pump"
CONTEXT_LAYER = "gold"
CONFIG_RUN_MODE = "test"
CONFIG_PROFILE = "default"
CONTEXT_LOG_FILE = "gold_model_replay_validation.log"

CTX = load_notebook_context(
    stage=CONTEXT_STAGE,
    dataset=CONTEXT_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    logger_child_name="capstone.gold.model_replay_validation",
    log_filename=CONTEXT_LOG_FILE,
)

paths = CTX.paths
CONFIG = CTX.config
CONFIG_MAP = CTX.config
STAGE_CFG = CTX.stage_config
RESOLVED_PATHS = CTX.resolved_paths
FILENAMES = CTX.filenames
RUNTIME_CFG = CTX.runtime
DATASET_CFG = CTX.dataset_config
PIPELINE = CTX.pipeline
logger = CTX.logger
ledger = CTX.ledger
LOG_PATH = CTX.log_path
CONTEXT_RECIPE_ID = CTX.recipe_id

DATASET_NAME = CONTEXT_DATASET
ARTIFACTS_ROOT = paths.artifacts
MODELS_ROOT = paths.models

logger.info(
    "Gold 06A replay validation context loaded",
    extra={
        "stage": CONTEXT_STAGE,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
        "log_path": str(LOG_PATH),
    },
)

ledger.add(
    kind="step",
    step="context_loaded",
    message="Loaded Gold 06A replay-validation context.",
    data={
        "stage": CONTEXT_STAGE,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-18 08:04:33,736 | INFO | capstone.gold.model_replay_validation | gold_model_replay_validation stage starting
2026-06-18 08:04:33,745 | INFO | capstone.gold.model_replay_validation | LEDGER | {'ts_utc': '2026-06-18T08:04:33.740897+00:00', 'stage': 'gold_model_replay_validation', 'recipe': 'gold_modeling__v001_test_replay_validation', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger from shared notebook context', 'why': None, 'consequence': None, 'data': {'stage': 'gold_model_replay_validation', 'recipe_id': 'gold_modeling__v001_test_replay_validation', 'dataset': 'pump', 'mode': 'test', 'profile': 'default', 'log_path': '/workspace/logs/gold_model_replay_validation.log'}}
2026-06-18 08:04:33,763 | INFO | capstone.gold.model_replay_validation | Gold 06A replay validation context loaded
2026-06-18 08:04:33,770 | INFO | capstone.gold.model_replay_validation | LEDGER | {'ts_utc': '2026-06-18T08:04:33.770170+00:00', 'stage': 'gold_model_replay_validation', 'recipe': 'go

{'ts_utc': '2026-06-18T08:04:33.770170+00:00',
 'stage': 'gold_model_replay_validation',
 'recipe': 'gold_modeling__v001_test_replay_validation',
 'kind': 'step',
 'step': 'context_loaded',
 'message': 'Loaded Gold 06A replay-validation context.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'gold_model_replay_validation',
  'dataset': 'pump',
  'mode': 'test',
  'profile': 'default',
  'log_path': '/workspace/logs/gold_model_replay_validation.log'}}

## 2. Resolve source artifacts and output folders

Gold 06A uses the saved artifacts from Gold 02, Gold 03A, Gold 03B, and Gold 03C. These artifacts include the fitted models, feature lists, thresholds, stage-rule settings, reference profiles, and summary metrics from the training notebooks.


In [14]:
GOLD_ROOT = ARTIFACTS_ROOT / "gold" / DATASET_NAME
VALIDATION_ROOT = GOLD_ROOT / "model_replay_validation"
VALIDATION_RESULTS_DIR = VALIDATION_ROOT / "results"
VALIDATION_SCORES_DIR = VALIDATION_ROOT / "scores"
VALIDATION_SUMMARY_DIR = VALIDATION_ROOT / "summaries"
VALIDATION_PLOTS_DIR = VALIDATION_ROOT / "plots"

for directory in [
    VALIDATION_ROOT,
    VALIDATION_RESULTS_DIR,
    VALIDATION_SCORES_DIR,
    VALIDATION_SUMMARY_DIR,
    VALIDATION_PLOTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Gold 01 split outputs. Prefer the full scaled dataframe because Stage 3 uses
# rolling/persistence/drift logic. The final metrics are still evaluated only on
# held-out test rows via meta__is_train_flag.
FULL_SCALED_PATH = Path(RESOLVED_PATHS.get(
    "gold_preprocessed_scaled_data_path",
    paths.data / "gold" / f"{DATASET_NAME}__gold__preprocessed_scaled.parquet",
))
TEST_DATA_PATH = Path(RESOLVED_PATHS.get(
    "gold_test_data_path",
    paths.data / "gold" / f"{DATASET_NAME}__gold__test.parquet",
))

FEATURE_DIR = GOLD_ROOT / "preprocessing" / "features"
STAGE1_FEATURES_PATH = FEATURE_DIR / f"{DATASET_NAME}__gold__stage1_features.json"
STAGE2_FEATURES_PATH = FEATURE_DIR / f"{DATASET_NAME}__gold__stage2_features.json"
STAGE3_PRIMARY_PATH = FEATURE_DIR / f"{DATASET_NAME}__gold__stage3_primary_rule_sensors.json"
STAGE3_SECONDARY_PATH = FEATURE_DIR / f"{DATASET_NAME}__gold__stage3_secondary_rule_sensors.json"

MODEL_ARTIFACTS = {
    "baseline": GOLD_ROOT / "baseline" / "models" / f"{DATASET_NAME}__gold__baseline_isolation_forest.joblib",
    "cascade_default_stage1": GOLD_ROOT / "cascade_defaults" / "models" / f"{DATASET_NAME}__gold__cascade_defaults_stage1_isolation_forest.joblib",
    "cascade_default_stage2": GOLD_ROOT / "cascade_defaults" / "models" / f"{DATASET_NAME}__gold__cascade_defaults_stage2_isolation_forest.joblib",
    "cascade_tuned_stage1": GOLD_ROOT / "cascade_tuned" / "models" / f"{DATASET_NAME}__gold__cascade_tuned_stage1_isolation_forest.joblib",
    "cascade_tuned_stage2": GOLD_ROOT / "cascade_tuned" / "models" / f"{DATASET_NAME}__gold__cascade_tuned_stage2_isolation_forest.joblib",
    "stage3_improved_stage1": GOLD_ROOT / "cascade_stage3_improved" / "models" / f"{DATASET_NAME}__gold__cascade_stage3_improved_stage1_isolation_forest.joblib",
    "stage3_improved_stage2": GOLD_ROOT / "cascade_stage3_improved" / "models" / f"{DATASET_NAME}__gold__cascade_stage3_improved_stage2_isolation_forest.joblib",
}

THRESHOLD_ARTIFACTS = {
    "baseline": GOLD_ROOT / "baseline" / "thresholds" / f"{DATASET_NAME}__gold__baseline_thresholds.json",
    "cascade_default": GOLD_ROOT / "cascade_defaults" / "thresholds" / f"{DATASET_NAME}__gold__cascade_defaults_thresholds.json",
    "cascade_tuned": GOLD_ROOT / "cascade_tuned" / "thresholds" / f"{DATASET_NAME}__gold__cascade_tuned_thresholds.json",
    "stage3_improved": GOLD_ROOT / "cascade_stage3_improved" / "thresholds" / f"{DATASET_NAME}__gold__cascade_stage3_improved_thresholds.json",
}

SUMMARY_ARTIFACTS = {
    "baseline": GOLD_ROOT / "baseline" / "summaries" / f"{DATASET_NAME}__gold__baseline_summary.json",
    "cascade_default": GOLD_ROOT / "cascade_defaults" / "summaries" / f"{DATASET_NAME}__gold__cascade_defaults_summary.json",
    "cascade_tuned": GOLD_ROOT / "cascade_tuned" / "summaries" / f"{DATASET_NAME}__gold__cascade_tuned_summary.json",
    "stage3_improved": GOLD_ROOT / "cascade_stage3_improved" / "summaries" / f"{DATASET_NAME}__gold__cascade_stage3_improved_summary.json",
}

PROFILE_ARTIFACTS = {
    "cascade_default": GOLD_ROOT / "cascade_defaults" / "profiles" / f"{DATASET_NAME}__gold__cascade_defaults_reference_profile.csv",
    "cascade_tuned": GOLD_ROOT / "cascade_tuned" / "profiles" / f"{DATASET_NAME}__gold__cascade_tuned_reference_profile.csv",
    "stage3_improved": GOLD_ROOT / "cascade_stage3_improved" / "profiles" / f"{DATASET_NAME}__gold__cascade_stage3_improved_reference_profile.csv",
}

CONFIG_ARTIFACTS = {
    "cascade_default": GOLD_ROOT / "cascade_defaults" / "config" / f"{DATASET_NAME}__gold_cascade_defaults__resolved_config.yaml",
    "cascade_tuned": GOLD_ROOT / "cascade_tuned" / "config" / f"{DATASET_NAME}__gold_cascade_tuned__resolved_config.yaml",
    "stage3_improved": GOLD_ROOT / "cascade_stage3_improved" / "config" / f"{DATASET_NAME}__gold_cascade_stage3_improved__resolved_config.yaml",
}

artifact_inventory = pd.DataFrame(
    [
        {"artifact_group": "model", "artifact_key": key, "path": str(path), "exists": path.exists()}
        for key, path in MODEL_ARTIFACTS.items()
    ]
    + [
        {"artifact_group": "threshold", "artifact_key": key, "path": str(path), "exists": path.exists()}
        for key, path in THRESHOLD_ARTIFACTS.items()
    ]
    + [
        {"artifact_group": "summary", "artifact_key": key, "path": str(path), "exists": path.exists()}
        for key, path in SUMMARY_ARTIFACTS.items()
    ]
    + [
        {"artifact_group": "profile", "artifact_key": key, "path": str(path), "exists": path.exists()}
        for key, path in PROFILE_ARTIFACTS.items()
    ]
    + [
        {"artifact_group": "feature", "artifact_key": path.stem, "path": str(path), "exists": path.exists()}
        for path in [STAGE1_FEATURES_PATH, STAGE2_FEATURES_PATH, STAGE3_PRIMARY_PATH, STAGE3_SECONDARY_PATH]
    ]
)

missing_artifacts = artifact_inventory.loc[~artifact_inventory["exists"]]
if len(missing_artifacts) > 0:
    raise FileNotFoundError(
        "Gold 06A cannot run because required artifacts are missing:\n"
        + missing_artifacts.to_string(index=False)
    )

ledger.add(
    kind="check",
    step="required_artifacts_found",
    message="Confirmed required Gold 06A model/config/feature/profile artifacts are available.",
    data={"artifact_count": int(len(artifact_inventory))},
    logger=logger,
)

display(artifact_inventory)


2026-06-18 08:04:34,426 | INFO | capstone.gold.model_replay_validation | LEDGER | {'ts_utc': '2026-06-18T08:04:34.426332+00:00', 'stage': 'gold_model_replay_validation', 'recipe': 'gold_modeling__v001_test_replay_validation', 'kind': 'check', 'step': 'required_artifacts_found', 'message': 'Confirmed required Gold 06A model/config/feature/profile artifacts are available.', 'why': None, 'consequence': None, 'data': {'artifact_count': 22}}


,artifact_group,artifact_key,path,exists
0,model,baseline,/workspace/artifacts/gold/pump/baseline/models...,True
1,model,cascade_default_stage1,/workspace/artifacts/gold/pump/cascade_default...,True
2,model,cascade_default_stage2,/workspace/artifacts/gold/pump/cascade_default...,True
3,model,cascade_tuned_stage1,/workspace/artifacts/gold/pump/cascade_tuned/m...,True
4,model,cascade_tuned_stage2,/workspace/artifacts/gold/pump/cascade_tuned/m...,True
5,model,stage3_improved_stage1,/workspace/artifacts/gold/pump/cascade_stage3_...,True
6,model,stage3_improved_stage2,/workspace/artifacts/gold/pump/cascade_stage3_...,True
7,threshold,baseline,/workspace/artifacts/gold/pump/baseline/thresh...,True
8,threshold,cascade_default,/workspace/artifacts/gold/pump/cascade_default...,True
9,threshold,cascade_tuned,/workspace/artifacts/gold/pump/cascade_tuned/t...,True


## 3. Load test source and training artifacts

The replay prefers the full scaled Gold dataframe because some Stage 3 rules use rolling windows and persistence. The evaluation mask still limits metric calculation to held-out test rows only.


In [15]:
if FULL_SCALED_PATH.exists():
    replay_source_dataframe = pd.read_parquet(FULL_SCALED_PATH)
    replay_source_name = "full_scaled_with_test_mask"
    if "meta__is_train_flag" not in replay_source_dataframe.columns:
        raise KeyError(
            "Full scaled dataframe exists but is missing meta__is_train_flag. "
            "Gold 06A needs this column to evaluate held-out test rows."
        )
    test_mask = ~replay_source_dataframe["meta__is_train_flag"].fillna(False).astype(bool)
else:
    # Fallback for portability. This can run the replay, but rolling Stage 3 rules
    # may not match the original notebook exactly because training rows are absent.
    replay_source_dataframe = pd.read_parquet(TEST_DATA_PATH)
    replay_source_name = "test_only_fallback"
    test_mask = pd.Series(True, index=replay_source_dataframe.index)

stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)
stage2_feature_columns = load_json(STAGE2_FEATURES_PATH)
stage3_primary_rule_sensors = load_json(STAGE3_PRIMARY_PATH)
stage3_secondary_rule_sensors = load_json(STAGE3_SECONDARY_PATH)

models = {key: joblib.load(path) for key, path in MODEL_ARTIFACTS.items()}
thresholds = {key: load_json(path) for key, path in THRESHOLD_ARTIFACTS.items()}
summaries = {key: load_json(path) for key, path in SUMMARY_ARTIFACTS.items()}
profiles = {key: pd.read_csv(path) for key, path in PROFILE_ARTIFACTS.items()}

config_payloads = {}
for key, path in CONFIG_ARTIFACTS.items():
    with path.open("r", encoding="utf-8") as file:
        config_payloads[key] = yaml.safe_load(file)

label_column_candidates = ["anomaly_flag", "is_anomaly", "target_flag", "label"]
label_column = next(
    (column for column in label_column_candidates if column in replay_source_dataframe.columns),
    None,
)
if label_column is None:
    raise KeyError(f"Could not find label column. Checked: {label_column_candidates}")

test_labels = replay_source_dataframe.loc[test_mask, label_column].fillna(0).astype(int).to_numpy(dtype=int)

basic_shape_check = {
    "replay_source_name": replay_source_name,
    "row_count": int(len(replay_source_dataframe)),
    "test_row_count": int(test_mask.sum()),
    "label_column": label_column,
    "test_anomaly_count": int(test_labels.sum()),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_sensor_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_sensor_count": int(len(stage3_secondary_rule_sensors)),
}

ledger.add(
    kind="check",
    step="replay_source_loaded",
    message="Loaded Gold replay source dataframe and model configuration artifacts.",
    data=basic_shape_check,
    logger=logger,
)

basic_shape_check


2026-06-18 08:04:39,830 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/preprocessing/features/pump__gold__stage1_features.json
2026-06-18 08:04:39,873 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/preprocessing/features/pump__gold__stage2_features.json
2026-06-18 08:04:39,940 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/preprocessing/features/pump__gold__stage3_primary_rule_sensors.json
2026-06-18 08:04:39,982 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/preprocessing/features/pump__gold__stage3_secondary_rule_sensors.json
2026-06-18 08:04:44,722 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/baseline/thresholds/pump__gold__baseline_thresholds.json
2026-06-18 08:04:44,754 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/cascade_defaults/thresholds/pump__gold__cascade_defaults_thresholds.json
2026-06-18 08:04:44,792 | INFO | capst

{'replay_source_name': 'full_scaled_with_test_mask',
 'row_count': 220320,
 'test_row_count': 83889,
 'label_column': 'anomaly_flag',
 'test_anomaly_count': 118,
 'stage1_feature_count': 50,
 'stage2_feature_count': 12,
 'stage3_primary_sensor_count': 5,
 'stage3_secondary_sensor_count': 8}

## 4. Replay helper functions

These functions intentionally mirror the core scoring logic from Gold 02, Gold 03A, Gold 03B, and Gold 03C without carrying over the training/search logic. The models and thresholds are already trained and selected; Gold 06A only reuses them.


In [16]:
def ensure_columns(
    dataframe: pd.DataFrame,
    columns: Sequence[str],
    *,
    context: str,
) -> None:
    """
    Confirm that a dataframe contains the requested columns.

    Parameters
    ----------
    dataframe:
        DataFrame being checked.
    columns:
        Column names required by the model or rule step.
    context:
        Human-readable label for the check, used in error messages.
    """
    missing_columns = [column for column in columns if column not in dataframe.columns]
    if missing_columns:
        raise KeyError(f"Missing columns for {context}: {missing_columns[:40]}")


def score_isolation_forest(
    model: Any,
    dataframe: pd.DataFrame,
    feature_columns: Sequence[str],
) -> dict[str, np.ndarray]:
    """
    Score a dataframe with an already-fitted Isolation Forest.

    The project uses `-score_samples` so larger values mean more anomalous.
    The helper also returns decision_function and predict outputs for review.
    """
    ensure_columns(dataframe, feature_columns, context="IsolationForest scoring")
    feature_frame = dataframe.loc[:, list(feature_columns)].copy()

    anomaly_score = -model.score_samples(feature_frame)
    decision = model.decision_function(feature_frame)
    prediction = model.predict(feature_frame)

    return {
        "score": anomaly_score,
        "decision": decision,
        "prediction": prediction,
    }


def compute_binary_metrics(
    *,
    dataframe: pd.DataFrame,
    test_mask: pd.Series,
    label_column: str,
    flag_column: str,
    score_column: str | None = None,
) -> dict[str, Any]:
    """
    Compute test-row alert counts and standard binary classification metrics.
    """
    test_dataframe = dataframe.loc[test_mask].copy()
    y_true = test_dataframe[label_column].fillna(0).astype(int).to_numpy(dtype=int)
    y_pred = test_dataframe[flag_column].fillna(0).astype(int).to_numpy(dtype=int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metrics: dict[str, Any] = {
        "alert_count_test_rows": int(y_pred.sum()),
        "test_rows": int(len(test_dataframe)),
        "test_anomaly_rows": int(y_true.sum()),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if score_column is not None and score_column in test_dataframe.columns and len(np.unique(y_true)) == 2:
        y_score = test_dataframe[score_column].fillna(0).astype(float).to_numpy(dtype=float)
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_score))
        metrics["pr_auc"] = float(average_precision_score(y_true, y_score))
    else:
        metrics["roc_auc"] = None
        metrics["pr_auc"] = None

    return metrics


def compute_profile_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: Sequence[str],
    output_name: str,
) -> pd.Series:
    """
    Count how many selected sensor features are outside their reference bounds.
    """
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns or feature_name not in reference_lookup:
            continue

        lower_bound = reference_lookup[feature_name]["lower_bound"]
        upper_bound = reference_lookup[feature_name]["upper_bound"]
        feature_breach_flag = (
            (dataframe[feature_name] < lower_bound)
            | (dataframe[feature_name] > upper_bound)
        ).astype(int)
        breach_counts = breach_counts + feature_breach_flag

    breach_counts.name = output_name
    return breach_counts


def compute_persistence_flag(
    flag_series: pd.Series,
    *,
    rolling_window_size: int,
    minimum_flags_in_window: int,
) -> pd.Series:
    """
    Mark rows where recent Stage 2 flags persist across a rolling window.
    """
    rolling_count = (
        flag_series.fillna(0)
        .astype(int)
        .rolling(window=rolling_window_size, min_periods=1)
        .sum()
    )
    return (rolling_count >= minimum_flags_in_window).astype(int)


def compute_drift_flag(
    dataframe: pd.DataFrame,
    *,
    feature_columns: Sequence[str],
    rolling_window_size: int,
    drift_threshold_multiplier: float,
) -> pd.Series:
    """
    Mark rows where any watched feature drifts away from its rolling median.
    """
    drift_trigger_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns:
            continue

        feature_series = dataframe[feature_name]
        feature_standard_deviation = feature_series.std()
        if pd.isna(feature_standard_deviation) or feature_standard_deviation == 0:
            continue

        rolling_median = feature_series.rolling(window=rolling_window_size, min_periods=1).median()
        rolling_delta = (feature_series - rolling_median).abs()
        feature_drift_flag = (
            rolling_delta > (feature_standard_deviation * drift_threshold_multiplier)
        ).astype(int)
        drift_trigger_counts = drift_trigger_counts + feature_drift_flag

    return (drift_trigger_counts >= 1).astype(int)


In [17]:
def add_stage3_broad_rules(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    stage3_config: Mapping[str, Any],
) -> pd.DataFrame:
    """
    Apply the broad Stage 3 confirmation rule used by Gold 03A and Gold 03B.
    """
    result = dataframe.copy()
    min_primary_hits = int(stage3_config["min_primary_sensor_hits"])
    min_secondary_hits = int(stage3_config["min_secondary_sensor_hits"])
    rolling_window_size = int(stage3_config["rolling_window_size"])
    minimum_flags_in_window = int(stage3_config["minimum_flags_in_window"])

    watch_features = list(dict.fromkeys(stage3_primary_rule_sensors + stage3_secondary_rule_sensors))

    result["stage3_profile_breach_count"] = compute_profile_breach_count(
        result,
        reference_profile=reference_profile,
        feature_columns=stage3_primary_rule_sensors,
        output_name="stage3_profile_breach_count",
    )
    result["stage3_secondary_breach_count"] = compute_profile_breach_count(
        result,
        reference_profile=reference_profile,
        feature_columns=stage3_secondary_rule_sensors,
        output_name="stage3_secondary_breach_count",
    )
    result["stage3_profile_breach_flag"] = (
        result["stage3_profile_breach_count"] >= min_primary_hits
    ).astype(int)
    result["stage3_corroboration_flag"] = (
        result["stage3_secondary_breach_count"] >= min_secondary_hits
    ).astype(int)
    result["stage3_persistence_flag"] = compute_persistence_flag(
        result["stage2_flag"],
        rolling_window_size=rolling_window_size,
        minimum_flags_in_window=minimum_flags_in_window,
    )
    result["stage3_drift_flag"] = compute_drift_flag(
        result,
        feature_columns=watch_features,
        rolling_window_size=5,
        drift_threshold_multiplier=1.0,
    )
    result["stage3_rule_evidence_count"] = (
        result["stage3_profile_breach_flag"]
        + result["stage3_persistence_flag"]
        + result["stage3_drift_flag"]
        + result["stage3_corroboration_flag"]
    )
    result["cascade_final_flag"] = (
        (result["stage1_flag"] == 1)
        & (result["stage2_flag"] == 1)
        & (
            (result["stage3_profile_breach_count"] >= min_primary_hits)
            | (result["stage3_rule_evidence_count"] >= 2)
        )
    ).astype(int)

    return result


def add_stage3_improved_rules(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    thresholds_payload: Mapping[str, Any],
    stage3_config: Mapping[str, Any],
) -> pd.DataFrame:
    """
    Apply the tuned Stage 3 confirmation rules used by Gold 03C.
    """
    result = dataframe.copy()
    selected_params = dict(thresholds_payload["stage3_selected_params"])

    min_primary_hits = int(stage3_config["min_primary_sensor_hits"])
    min_secondary_hits = int(stage3_config["min_secondary_sensor_hits"])
    min_weighted_score = float(selected_params["min_weighted_score"])
    rolling_window_size = int(selected_params["rolling_window_size"])
    minimum_flags_in_window = int(selected_params["minimum_flags_in_window"])
    strong_primary_hits = int(selected_params["strong_primary_hits"])
    drift_threshold_multiplier = float(selected_params["drift_threshold_multiplier"])

    profile_weight = float(thresholds_payload.get("stage3_profile_breach_weight", 2.0))
    corroboration_weight = float(thresholds_payload.get("stage3_corroboration_weight", 1.0))
    persistence_weight = float(thresholds_payload.get("stage3_persistence_weight", 1.0))
    drift_weight = float(thresholds_payload.get("stage3_drift_weight", 1.0))
    drift_rolling_window_size = int(thresholds_payload.get("stage3_drift_rolling_window_size", 5))
    watch_features = list(dict.fromkeys(stage3_primary_rule_sensors + stage3_secondary_rule_sensors))

    result["stage3_profile_breach_count"] = compute_profile_breach_count(
        result,
        reference_profile=reference_profile,
        feature_columns=stage3_primary_rule_sensors,
        output_name="stage3_profile_breach_count",
    )
    result["stage3_secondary_breach_count"] = compute_profile_breach_count(
        result,
        reference_profile=reference_profile,
        feature_columns=stage3_secondary_rule_sensors,
        output_name="stage3_secondary_breach_count",
    )
    result["stage3_profile_breach_flag"] = (
        result["stage3_profile_breach_count"] >= min_primary_hits
    ).astype(int)
    result["stage3_strong_primary_flag"] = (
        result["stage3_profile_breach_count"] >= strong_primary_hits
    ).astype(int)
    result["stage3_corroboration_flag"] = (
        result["stage3_secondary_breach_count"] >= min_secondary_hits
    ).astype(int)
    result["stage3_persistence_flag"] = compute_persistence_flag(
        result["stage2_flag"],
        rolling_window_size=rolling_window_size,
        minimum_flags_in_window=minimum_flags_in_window,
    )
    result["stage3_drift_flag"] = compute_drift_flag(
        result,
        feature_columns=watch_features,
        rolling_window_size=drift_rolling_window_size,
        drift_threshold_multiplier=drift_threshold_multiplier,
    )
    result["stage3_rule_evidence_count"] = (
        result["stage3_profile_breach_flag"]
        + result["stage3_persistence_flag"]
        + result["stage3_drift_flag"]
        + result["stage3_corroboration_flag"]
    )
    result["stage3_weighted_evidence_score"] = (
        result["stage3_profile_breach_flag"] * profile_weight
        + result["stage3_corroboration_flag"] * corroboration_weight
        + result["stage3_persistence_flag"] * persistence_weight
        + result["stage3_drift_flag"] * drift_weight
    )
    result["stage3_weighted_score"] = result["stage3_weighted_evidence_score"]
    result["stage3_confirmed_flag"] = (
        (result["stage3_strong_primary_flag"] == 1)
        | (
            (result["stage3_profile_breach_flag"] == 1)
            & (result["stage3_weighted_evidence_score"] >= min_weighted_score)
        )
    ).astype(int)

    result["cascade_stage3_improved_flag"] = (
        (result["stage2_flag"] == 1)
        & (result["stage3_confirmed_flag"] == 1)
    ).astype(int)
    result["cascade_final_flag"] = result["cascade_stage3_improved_flag"].astype(int)

    result["cascade_stage3_relaxed_flag"] = (
        (result["stage2_flag"] == 1)
        & (result["stage3_weighted_evidence_score"] >= 2.0)
    ).astype(int)
    result["cascade_stage3_medium_flag"] = (
        (result["stage2_flag"] == 1)
        & (result["stage3_weighted_evidence_score"] >= 3.0)
    ).astype(int)
    result["cascade_stage3_strict_flag"] = (
        (result["stage2_flag"] == 1)
        & (result["stage3_weighted_evidence_score"] >= 5.0)
    ).astype(int)

    return result


## 5. Replay baseline and cascade variants

This section is the core validation step. It reuses the saved model and rule artifacts, applies them to the replay source, and keeps only held-out test rows for metric comparison.


In [18]:
def run_baseline_replay(source_dataframe: pd.DataFrame) -> pd.DataFrame:
    """Replay the Gold 02 baseline Isolation Forest using the saved threshold."""
    result = source_dataframe.copy()
    score_payload = score_isolation_forest(
        models["baseline"],
        result,
        stage1_feature_columns,
    )
    threshold = float(thresholds["baseline"]["baseline_threshold"])
    result["baseline_score"] = score_payload["score"]
    result["baseline_decision"] = score_payload["decision"]
    result["baseline_pred"] = score_payload["prediction"]
    result["baseline_threshold"] = threshold
    result["baseline_flag"] = (result["baseline_score"] >= threshold).astype(int)
    return result


def run_cascade_replay(
    source_dataframe: pd.DataFrame,
    *,
    variant_key: str,
    stage1_model_key: str,
    stage2_model_key: str,
    use_improved_stage3: bool,
) -> pd.DataFrame:
    """Replay one saved cascade variant against the replay source dataframe."""
    result = source_dataframe.copy()
    threshold_payload = thresholds[variant_key]
    stage3_config = config_payloads[variant_key]["gold_cascade"]["stage3"]

    stage1_scores = score_isolation_forest(models[stage1_model_key], result, stage1_feature_columns)
    stage2_scores = score_isolation_forest(models[stage2_model_key], result, stage2_feature_columns)

    stage1_threshold = float(threshold_payload["stage1_threshold"])
    stage2_threshold = float(threshold_payload["stage2_threshold"])

    result["stage1_score"] = stage1_scores["score"]
    result["stage1_decision"] = stage1_scores["decision"]
    result["stage1_pred"] = stage1_scores["prediction"]
    result["stage1_threshold"] = stage1_threshold
    result["stage1_flag"] = (result["stage1_score"] >= stage1_threshold).astype(int)

    result["stage2_score"] = stage2_scores["score"]
    result["stage2_model_score"] = stage2_scores["score"]
    result["stage2_model_decision"] = stage2_scores["decision"]
    result["stage2_model_pred"] = stage2_scores["prediction"]
    result["stage2_threshold"] = stage2_threshold
    result["stage2_raw_flag"] = (result["stage2_score"] >= stage2_threshold).astype(int)
    result["stage2_model_flag"] = result["stage2_raw_flag"].astype(int)
    result["stage2_flag"] = (
        (result["stage1_flag"] == 1)
        & (result["stage2_raw_flag"] == 1)
    ).astype(int)

    if use_improved_stage3:
        return add_stage3_improved_rules(
            result,
            reference_profile=profiles[variant_key],
            thresholds_payload=threshold_payload,
            stage3_config=stage3_config,
        )

    return add_stage3_broad_rules(
        result,
        reference_profile=profiles[variant_key],
        stage3_config=stage3_config,
    )

baseline_replay = run_baseline_replay(replay_source_dataframe)
cascade_default_replay = run_cascade_replay(
    replay_source_dataframe,
    variant_key="cascade_default",
    stage1_model_key="cascade_default_stage1",
    stage2_model_key="cascade_default_stage2",
    use_improved_stage3=False,
)
cascade_tuned_replay = run_cascade_replay(
    replay_source_dataframe,
    variant_key="cascade_tuned",
    stage1_model_key="cascade_tuned_stage1",
    stage2_model_key="cascade_tuned_stage2",
    use_improved_stage3=False,
)
stage3_improved_replay = run_cascade_replay(
    replay_source_dataframe,
    variant_key="stage3_improved",
    stage1_model_key="stage3_improved_stage1",
    stage2_model_key="stage3_improved_stage2",
    use_improved_stage3=True,
)

ledger.add(
    kind="step",
    step="model_variants_replayed",
    message="Replayed saved baseline and cascade model artifacts against the Gold test split source.",
    data={
        "baseline_rows": int(len(baseline_replay)),
        "cascade_default_rows": int(len(cascade_default_replay)),
        "cascade_tuned_rows": int(len(cascade_tuned_replay)),
        "stage3_improved_rows": int(len(stage3_improved_replay)),
        "test_rows": int(test_mask.sum()),
    },
    logger=logger,
)

print("Replay complete.")


2026-06-18 08:06:13,814 | INFO | capstone.gold.model_replay_validation | LEDGER | {'ts_utc': '2026-06-18T08:06:13.814422+00:00', 'stage': 'gold_model_replay_validation', 'recipe': 'gold_modeling__v001_test_replay_validation', 'kind': 'step', 'step': 'model_variants_replayed', 'message': 'Replayed saved baseline and cascade model artifacts against the Gold test split source.', 'why': None, 'consequence': None, 'data': {'baseline_rows': 220320, 'cascade_default_rows': 220320, 'cascade_tuned_rows': 220320, 'stage3_improved_rows': 220320, 'test_rows': 83889}}


Replay complete.


## 6. Build replay metrics and compare to training-run artifacts

This section evaluates only the held-out test rows and compares those replayed metrics to the metrics saved by Gold 02, Gold 03A, Gold 03B, and Gold 03C.

The comparison is performed in two layers. First, the notebook performs a strict exact-match check. A row receives `validation_status = "pass"` only when the replayed alert count, precision, recall, and F1 match the saved training artifact values exactly.

If a row does not pass the exact check, the notebook then applies a small tolerance check. The tolerance allows an alert-count difference of no more than one row and metric differences no greater than 0.0001. This captures threshold-boundary or rolling-rule edge cases while still preserving the exact deltas in the output table.

The final reportable status is stored in `final_validation_status`.

In [19]:
replay_metric_specs = [
    {
        "model_id": "baseline",
        "model_label": "Baseline IsolationForest",
        "source_notebook": "gold_02",
        "dataframe": baseline_replay,
        "flag_column": "baseline_flag",
        "score_column": "baseline_score",
    },
    {
        "model_id": "cascade_default",
        "model_label": "Cascade Default",
        "source_notebook": "gold_03a",
        "dataframe": cascade_default_replay,
        "flag_column": "cascade_final_flag",
        "score_column": "stage2_score",
    },
    {
        "model_id": "cascade_tuned",
        "model_label": "Cascade Tuned",
        "source_notebook": "gold_03b",
        "dataframe": cascade_tuned_replay,
        "flag_column": "cascade_final_flag",
        "score_column": "stage2_score",
    },
    {
        "model_id": "stage3_improved",
        "model_label": "Stage 3 Improved",
        "source_notebook": "gold_03c",
        "dataframe": stage3_improved_replay,
        "flag_column": "cascade_final_flag",
        "score_column": "stage3_weighted_score",
    },
    {
        "model_id": "stage3_relaxed",
        "model_label": "Stage 3 Relaxed",
        "source_notebook": "gold_03c",
        "dataframe": stage3_improved_replay,
        "flag_column": "cascade_stage3_relaxed_flag",
        "score_column": "stage3_weighted_score",
    },
    {
        "model_id": "stage3_medium",
        "model_label": "Stage 3 Medium",
        "source_notebook": "gold_03c",
        "dataframe": stage3_improved_replay,
        "flag_column": "cascade_stage3_medium_flag",
        "score_column": "stage3_weighted_score",
    },
    {
        "model_id": "stage3_strict",
        "model_label": "Stage 3 Strict",
        "source_notebook": "gold_03c",
        "dataframe": stage3_improved_replay,
        "flag_column": "cascade_stage3_strict_flag",
        "score_column": "stage3_weighted_score",
    },
]

replay_metric_rows = []
for spec in replay_metric_specs:
    metrics = compute_binary_metrics(
        dataframe=spec["dataframe"],
        test_mask=test_mask,
        label_column=label_column,
        flag_column=spec["flag_column"],
        score_column=spec["score_column"],
    )
    replay_metric_rows.append(
        {
            "model_id": spec["model_id"],
            "model_label": spec["model_label"],
            "source_notebook": spec["source_notebook"],
            "flag_column": spec["flag_column"],
            "score_column": spec["score_column"],
            **metrics,
        }
    )

replay_metrics_dataframe = pd.DataFrame(replay_metric_rows)

def build_expected_metrics_from_training_artifacts() -> pd.DataFrame:
    """Collect the saved test-row metrics from the Gold 02/03 training artifacts."""
    rows: list[dict[str, Any]] = []

    baseline_metrics = summaries["baseline"]["baseline_metrics"]
    rows.append({
        "model_id": "baseline",
        "expected_alert_count_test_rows": int(baseline_metrics["alert_count_test_rows"]),
        "expected_precision": float(baseline_metrics["precision"]),
        "expected_recall": float(baseline_metrics["recall"]),
        "expected_f1": float(baseline_metrics["f1"]),
        "expected_source": str(SUMMARY_ARTIFACTS["baseline"]),
    })

    for model_id, summary_key in [
        ("cascade_default", "cascade_default"),
        ("cascade_tuned", "cascade_tuned"),
        ("stage3_improved", "stage3_improved"),
    ]:
        cascade_metrics = summaries[summary_key]["cascade_metrics"]
        rows.append({
            "model_id": model_id,
            "expected_alert_count_test_rows": int(cascade_metrics["final_alert_count_test_rows"]),
            "expected_precision": float(cascade_metrics["precision"]),
            "expected_recall": float(cascade_metrics["recall"]),
            "expected_f1": float(cascade_metrics["f1"]),
            "expected_source": str(SUMMARY_ARTIFACTS[summary_key]),
        })

    stage3_metrics = summaries["stage3_improved"]["cascade_metrics"]
    for mode_name in ["relaxed", "medium", "strict"]:
        rows.append({
            "model_id": f"stage3_{mode_name}",
            "expected_alert_count_test_rows": int(
                summaries["stage3_improved"][f"stage3_{mode_name}_alert_count_test_rows"]
            ),
            "expected_precision": float(stage3_metrics[f"stage3_{mode_name}_precision"]),
            "expected_recall": float(stage3_metrics[f"stage3_{mode_name}_recall"]),
            "expected_f1": float(stage3_metrics[f"stage3_{mode_name}_f1"]),
            "expected_source": str(SUMMARY_ARTIFACTS["stage3_improved"]),
        })

    return pd.DataFrame(rows)

expected_metrics_dataframe = build_expected_metrics_from_training_artifacts()

replay_comparison_dataframe = replay_metrics_dataframe.merge(
    expected_metrics_dataframe,
    on="model_id",
    how="left",
)

for metric_name in ["alert_count_test_rows", "precision", "recall", "f1"]:
    actual_column = metric_name
    expected_column = f"expected_{metric_name}"
    delta_column = f"{metric_name}_delta"
    if actual_column in replay_comparison_dataframe.columns and expected_column in replay_comparison_dataframe.columns:
        replay_comparison_dataframe[delta_column] = (
            replay_comparison_dataframe[actual_column]
            - replay_comparison_dataframe[expected_column]
        )

METRIC_TOLERANCE = 1e-9
replay_comparison_dataframe["alert_count_match"] = (
    replay_comparison_dataframe["alert_count_test_rows_delta"].fillna(np.inf).abs() == 0
)
replay_comparison_dataframe["precision_match"] = (
    replay_comparison_dataframe["precision_delta"].fillna(np.inf).abs() <= METRIC_TOLERANCE
)
replay_comparison_dataframe["recall_match"] = (
    replay_comparison_dataframe["recall_delta"].fillna(np.inf).abs() <= METRIC_TOLERANCE
)
replay_comparison_dataframe["f1_match"] = (
    replay_comparison_dataframe["f1_delta"].fillna(np.inf).abs() <= METRIC_TOLERANCE
)
replay_comparison_dataframe["validation_status"] = np.where(
    replay_comparison_dataframe[["alert_count_match", "precision_match", "recall_match", "f1_match"]].all(axis=1),
    "pass",
    "review_delta",
)


# ---------------------------------------------------------
# Secondary tolerance check for near-exact replay matches
# ---------------------------------------------------------
# The exact validation_status above remains the strict check.
# If every metric matches exactly, validation_status stays "pass".
#
# If a row does not pass exactly, this second check determines whether the row
# is still close enough to be considered a practical replay match.
#
# Current tolerance rule:
# - alert count may differ by no more than 1 row;
# - precision, recall, and F1 may differ by no more than 0.0001.
#
# This preserves both interpretations:
# - validation_status = strict exact-match result
# - tolerance_validation_status = secondary within-tolerance result
# - final_validation_status = reportable final result

ALERT_COUNT_TOLERANCE = 1
METRIC_TOLERANCE_RELAXED = 0.0001

replay_comparison_dataframe["alert_count_within_tolerance"] = (
    replay_comparison_dataframe["alert_count_test_rows_delta"]
    .fillna(np.inf)
    .abs()
    <= ALERT_COUNT_TOLERANCE
)

replay_comparison_dataframe["precision_within_tolerance"] = (
    replay_comparison_dataframe["precision_delta"]
    .fillna(np.inf)
    .abs()
    <= METRIC_TOLERANCE_RELAXED
)

replay_comparison_dataframe["recall_within_tolerance"] = (
    replay_comparison_dataframe["recall_delta"]
    .fillna(np.inf)
    .abs()
    <= METRIC_TOLERANCE_RELAXED
)

replay_comparison_dataframe["f1_within_tolerance"] = (
    replay_comparison_dataframe["f1_delta"]
    .fillna(np.inf)
    .abs()
    <= METRIC_TOLERANCE_RELAXED
)

replay_comparison_dataframe["tolerance_validation_status"] = np.where(
    replay_comparison_dataframe[
        [
            "alert_count_within_tolerance",
            "precision_within_tolerance",
            "recall_within_tolerance",
            "f1_within_tolerance",
        ]
    ].all(axis=1),
    "pass_with_tolerance",
    "review_delta",
)

replay_comparison_dataframe["final_validation_status"] = np.where(
    replay_comparison_dataframe["validation_status"].eq("pass"),
    "exact_pass",
    replay_comparison_dataframe["tolerance_validation_status"],
)


display(replay_comparison_dataframe)


,model_id,model_label,source_notebook,flag_column,score_column,alert_count_test_rows,test_rows,test_anomaly_rows,precision,recall,...,precision_match,recall_match,f1_match,validation_status,alert_count_within_tolerance,precision_within_tolerance,recall_within_tolerance,f1_within_tolerance,tolerance_validation_status,final_validation_status
0,baseline,Baseline IsolationForest,gold_02,baseline_flag,baseline_score,31200,83889,118,0.003782,1.000000,...,True,True,True,pass,True,True,True,True,pass_with_tolerance,exact_pass
1,cascade_default,Cascade Default,gold_03a,cascade_final_flag,stage2_score,24896,83889,118,0.003093,0.652542,...,False,True,False,review_delta,True,True,True,True,pass_with_tolerance,pass_with_tolerance
2,cascade_tuned,Cascade Tuned,gold_03b,cascade_final_flag,stage2_score,15153,83889,118,0.005345,0.686441,...,True,True,True,pass,True,True,True,True,pass_with_tolerance,exact_pass
3,stage3_improved,Stage 3 Improved,gold_03c,cascade_final_flag,stage3_weighted_score,6594,83889,118,0.010616,0.593220,...,True,True,True,pass,True,True,True,True,pass_with_tolerance,exact_pass
4,stage3_relaxed,Stage 3 Relaxed,gold_03c,cascade_stage3_relaxed_flag,stage3_weighted_score,13713,83889,118,0.005834,0.677966,...,True,True,True,pass,True,True,True,True,pass_with_tolerance,exact_pass
5,stage3_medium,Stage 3 Medium,gold_03c,cascade_stage3_medium_flag,stage3_weighted_score,7287,83889,118,0.010704,0.661017,...,False,True,False,review_delta,True,True,True,True,pass_with_tolerance,pass_with_tolerance
6,stage3_strict,Stage 3 Strict,gold_03c,cascade_stage3_strict_flag,stage3_weighted_score,61,83889,118,0.442623,0.228814,...,True,True,True,pass,True,True,True,True,pass_with_tolerance,exact_pass


## 7. Save Gold 06A outputs

The saved row-level replay output is intentionally limited to useful identifier, label, score, and flag columns. Gold 06B uses this file for early-warning validation on the replayed test outputs.


In [20]:
# Build one wide test-row dataframe with final replay outputs from all model variants.
identity_columns = [
    column for column in [
        "meta__row_id",
        "meta__asset_id",
        "meta__dataset",
        "meta__run_id",
        "meta__split",
        "meta__is_train_flag",
        "event_time",
        "event_step",
        "time_index",
        "timestamp",
        "machine_status",
        label_column,
    ]
    if column in replay_source_dataframe.columns
]

replay_scores_dataframe = replay_source_dataframe.loc[test_mask, identity_columns].copy()
replay_scores_dataframe = replay_scores_dataframe.reset_index(drop=True)
replay_scores_dataframe["plot_order_index"] = np.arange(len(replay_scores_dataframe), dtype=int)

score_sources = {
    "baseline": (baseline_replay, ["baseline_score", "baseline_flag"]),
    "cascade_default": (cascade_default_replay, ["stage1_score", "stage1_flag", "stage2_score", "stage2_raw_flag", "stage2_flag", "cascade_final_flag"]),
    "cascade_tuned": (cascade_tuned_replay, ["stage1_score", "stage1_flag", "stage2_score", "stage2_raw_flag", "stage2_flag", "cascade_final_flag"]),
    "stage3_improved": (stage3_improved_replay, [
        "stage1_score", "stage1_flag", "stage2_score", "stage2_raw_flag", "stage2_flag",
        "stage3_weighted_score", "cascade_final_flag", "cascade_stage3_relaxed_flag",
        "cascade_stage3_medium_flag", "cascade_stage3_strict_flag",
    ]),
}

for prefix, (source_df, columns) in score_sources.items():
    for column in columns:
        if column in source_df.columns:
            replay_scores_dataframe[f"{prefix}__{column}"] = (
                source_df.loc[test_mask, column].reset_index(drop=True)
            )

metrics_output_path = VALIDATION_RESULTS_DIR / f"{DATASET_NAME}__gold06a__test_replay_metrics.csv"
comparison_output_path = VALIDATION_RESULTS_DIR / f"{DATASET_NAME}__gold06a__test_replay_vs_training_artifacts.csv"
scores_output_path = VALIDATION_SCORES_DIR / f"{DATASET_NAME}__gold06a__test_replay_scores.csv"
summary_output_path = VALIDATION_SUMMARY_DIR / f"{DATASET_NAME}__gold06a__test_replay_summary.json"

replay_metrics_dataframe.to_csv(metrics_output_path, index=False)
replay_comparison_dataframe.to_csv(comparison_output_path, index=False)
replay_scores_dataframe.to_csv(scores_output_path, index=False)

summary_payload = {
    "stage": CONTEXT_STAGE,
    "dataset": DATASET_NAME,
    "recipe_id": CONTEXT_RECIPE_ID,
    "replay_source_name": replay_source_name,
    "replay_source_rows": int(len(replay_source_dataframe)),
    "test_rows": int(test_mask.sum()),
    "label_column": label_column,
    "model_count": int(len(replay_metric_specs)),
    "exact_pass_count": int(
        replay_comparison_dataframe["final_validation_status"]
        .eq("exact_pass")
        .sum()
    ),
    "tolerance_pass_count": int(
        replay_comparison_dataframe["final_validation_status"]
        .eq("pass_with_tolerance")
        .sum()
    ),
    "pass_count": int(
        replay_comparison_dataframe["final_validation_status"]
        .isin(["exact_pass", "pass_with_tolerance"])
        .sum()
    ),
    "review_count": int(
        replay_comparison_dataframe["final_validation_status"]
        .eq("review_delta")
        .sum()
    ),
    "alert_count_tolerance": ALERT_COUNT_TOLERANCE,
    "metric_tolerance_relaxed": METRIC_TOLERANCE_RELAXED,
    "metrics_output_path": str(metrics_output_path),
    "comparison_output_path": str(comparison_output_path),
    "scores_output_path": str(scores_output_path),
    "note": (
        "Gold 06A replays saved Gold 02/03 model and rule artifacts against held-out test rows. "
        "It does not retrain models. It compares replayed test metrics to the metrics saved by the training notebooks."
    ),
}

save_json(summary_payload, summary_output_path)

ledger.add(
    kind="result",
    step="gold06a_outputs_saved",
    message="Saved Gold 06A replay metrics, replay-vs-training comparison, row-level test replay scores, and summary JSON.",
    data=summary_payload,
    logger=logger,
)

metrics_output_path, comparison_output_path, scores_output_path, summary_output_path


2026-06-18 08:06:37,158 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/model_replay_validation/summaries/pump__gold06a__test_replay_summary.json
2026-06-18 08:06:37,162 | INFO | capstone.gold.model_replay_validation | LEDGER | {'ts_utc': '2026-06-18T08:06:37.162445+00:00', 'stage': 'gold_model_replay_validation', 'recipe': 'gold_modeling__v001_test_replay_validation', 'kind': 'result', 'step': 'gold06a_outputs_saved', 'message': 'Saved Gold 06A replay metrics, replay-vs-training comparison, row-level test replay scores, and summary JSON.', 'why': None, 'consequence': None, 'data': {'stage': 'gold_model_replay_validation', 'dataset': 'pump', 'recipe_id': 'gold_modeling__v001_test_replay_validation', 'replay_source_name': 'full_scaled_with_test_mask', 'replay_source_rows': 220320, 'test_rows': 83889, 'label_column': 'anomaly_flag', 'model_count': 7, 'exact_pass_count': 5, 'tolerance_pass_count': 2, 'pass_count': 7, 'review_count': 0, 'alert_count_tolerance': 1, 'm

(PosixPath('/workspace/artifacts/gold/pump/model_replay_validation/results/pump__gold06a__test_replay_metrics.csv'),
 PosixPath('/workspace/artifacts/gold/pump/model_replay_validation/results/pump__gold06a__test_replay_vs_training_artifacts.csv'),
 PosixPath('/workspace/artifacts/gold/pump/model_replay_validation/scores/pump__gold06a__test_replay_scores.csv'),
 PosixPath('/workspace/artifacts/gold/pump/model_replay_validation/summaries/pump__gold06a__test_replay_summary.json'))

## 8. Gold 06A interpretation

Use this final object as the notebook-level interpretation for Gold 06A.


In [21]:
interpretation = {
    "purpose": (
        "Gold 06A validates that saved Gold 02/03 model artifacts and rule settings can be replayed "
        "against held-out test rows without retraining."
    ),
    "validation_scope": (
        "Baseline, cascade default, cascade tuned, Stage 3 improved, and Stage 3 relaxed/medium/strict outputs."
    ),
    "comparison_scope": (
        "Replay metrics are compared against the saved training-notebook summary artifacts, not against Gold 04 as the primary benchmark."
    ),
    "exact_pass_count": int(
    replay_comparison_dataframe["final_validation_status"]
    .eq("exact_pass")
    .sum()
    ),
    "tolerance_pass_count": int(
        replay_comparison_dataframe["final_validation_status"]
        .eq("pass_with_tolerance")
        .sum()
    ),
    "pass_count": int(
        replay_comparison_dataframe["final_validation_status"]
        .isin(["exact_pass", "pass_with_tolerance"])
        .sum()
    ),
    "review_count": int(
        replay_comparison_dataframe["final_validation_status"]
        .eq("review_delta")
        .sum()
    ),
    "tolerance_note": (
        "Rows that fail the exact replay check are evaluated with a small tolerance: "
        "alert-count delta <= 1 and precision/recall/F1 deltas <= 0.0001."
    ),
    "next_notebook": "Gold 06B uses the saved Gold 06A replay score output for test-set early-warning validation.",
}

display(interpretation)


{'purpose': 'Gold 06A validates that saved Gold 02/03 model artifacts and rule settings can be replayed against held-out test rows without retraining.',
 'validation_scope': 'Baseline, cascade default, cascade tuned, Stage 3 improved, and Stage 3 relaxed/medium/strict outputs.',
 'comparison_scope': 'Replay metrics are compared against the saved training-notebook summary artifacts, not against Gold 04 as the primary benchmark.',
 'exact_pass_count': 5,
 'tolerance_pass_count': 2,
 'pass_count': 7,
 'review_count': 0,
 'tolerance_note': 'Rows that fail the exact replay check are evaluated with a small tolerance: alert-count delta <= 1 and precision/recall/F1 deltas <= 0.0001.',
 'next_notebook': 'Gold 06B uses the saved Gold 06A replay score output for test-set early-warning validation.'}